# AnomalyCLIP adversarial robustness on MVTec AD

This notebook is orchestration-only. It clones the public adversarial-robustness repository and the official AnomalyCLIP repository, installs dependencies, configures the standalone adversarial harness, runs it, and packages the outputs. All attack, model, dataset, metric, and artifact code is loaded from the cloned adversarial-robustness repository.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

print('===== STEP 1: CLONE/UPDATE REPOSITORIES =====')
working = Path('/kaggle/working')
repo_root = working / 'adversarial-robustness'
anomalyclip_root = working / 'AnomalyCLIP'

repo_url = 'https://github.com/Parsagh05/adversarial-robustness.git'
if repo_root.exists():
    subprocess.run(['git', '-C', str(repo_root), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', repo_url, str(repo_root)], check=True)

official_url = 'https://github.com/zqhang/AnomalyCLIP.git'
if anomalyclip_root.exists():
    subprocess.run(['git', '-C', str(anomalyclip_root), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', official_url, str(anomalyclip_root)], check=True)

experiment_root = repo_root
requirements = experiment_root / 'requirements.txt'
print('===== STEP 2: INSTALL DEPENDENCIES =====')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)], check=True)
print(f'Experiment code: {experiment_root}')
print(f'Official AnomalyCLIP: {anomalyclip_root}')
print('Environment ready.')

In [ ]:
import os
import sys
from pathlib import Path
import torch

experiment_root = Path('/kaggle/working/adversarial-robustness')
if str(experiment_root) not in sys.path:
    sys.path.insert(0, str(experiment_root))

from adversarial_harness import AttackConfig, ExperimentConfig, run_experiment

print('===== STEP 3: RESOLVE DATA AND CHECKPOINT =====')
MVTEC_ROOT = Path('/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection')
ANOMALYCLIP_ROOT = Path('/kaggle/working/AnomalyCLIP')

# The official repository's 9_12_4_multiscale checkpoint was trained on
# VisA and is therefore the correct zero-shot checkpoint for MVTec. The
# similarly named 9_12_4_multiscale_visa directory was trained on MVTec.
CHECKPOINT_PATH = (
    ANOMALYCLIP_ROOT
    / 'checkpoints'
    / '9_12_4_multiscale'
    / 'epoch_15.pth'
)
if not CHECKPOINT_PATH.is_file():
    available = sorted((ANOMALYCLIP_ROOT / 'checkpoints').rglob('*.pth'))
    raise FileNotFoundError(
        f'Official VisA-trained checkpoint was not found: {CHECKPOINT_PATH}. '
        f'Available official checkpoints: {available}'
    )

if not MVTEC_ROOT.is_dir():
    raise FileNotFoundError(f'MVTec mount not found: {MVTEC_ROOT}')
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before running this experiment.')

os.environ['ANOMALYCLIP_ROOT'] = str(ANOMALYCLIP_ROOT)
os.environ['ANOMALYCLIP_CHECKPOINT'] = str(CHECKPOINT_PATH)
os.environ['ANOMALYCLIP_CLIP_CACHE'] = '/kaggle/working/clip_cache'

# False gives a small end-to-end validation. Change to True for the full
# 2 directions x 3 objectives x 3 scopes benchmark (18 conditions).
FULL_RUN = True
OUTPUT_ROOT = Path(
    '/kaggle/working/anomalyclip_adversarial_held_out_full'
    if FULL_RUN else '/kaggle/working/anomalyclip_adversarial_held_out_smoke'
)
# Optional reusable artifact produced by kaggle_calibrate_anomalyclip_thresholds.ipynb.
# Leave as None to calibrate once inside OUTPUT_ROOT from MVTec train/good.
# THRESHOLDS_PATH = None
THRESHOLDS_PATH = Path(
    "/kaggle/input/datasets/parsagholami/thresholds/category_thresholds.json"
)


if FULL_RUN:
    categories = None
    scopes = ('per_image', 'per_category', 'dataset')
    directions = ('normal_to_abnormal', 'abnormal_to_normal')
    loss_modes = ('global', 'local', 'combined')
    steps = 20
    universal_steps = 200
    max_samples = None
    compute_lpips = True
else:
    categories = ('bottle',)
    scopes = ('per_image',)
    directions = ('abnormal_to_normal',)
    loss_modes = ('combined',)
    steps = 2
    universal_steps = 2
    max_samples = 4
    compute_lpips = False

attack = AttackConfig(
    image_size=518,
    epsilon=8 / 255,
    step_size=2 / 255,
    steps=steps,
    universal_steps=universal_steps,
    random_start=True,
    temperature=0.07,
    global_weight=0.5,
    local_weight=0.5,
    feature_layers=(6, 12, 18, 24),
    scopes=scopes,
    directions=directions,
    loss_modes=loss_modes,
    per_image_batch_size=1,
    universal_batch_size=2,
    seed=111,
)

config = ExperimentConfig(
    mvtec_root=str(MVTEC_ROOT),
    output_root=str(OUTPUT_ROOT),
    anomalyclip_root=str(ANOMALYCLIP_ROOT),
    anomalyclip_checkpoint=str(CHECKPOINT_PATH),
    device='cuda',
    target_model='AnomalyCLIP',
    categories=categories,
    target_batch_size=2,
    metric_size=518,
    aupro_fpr_limit=0.30,
    aupro_max_thresholds=200,
    anomaly_map_sigma=4.0,
    compute_lpips=compute_lpips,
    save_universal_perturbations=True,
    save_adversarial_examples=5 if FULL_RUN else 2,
    universal_protocol='held_out',
    fit_fraction=0.5,
    split_seed=111,
    diagnostic_max_samples=64,
    threshold_mode='normal_train_quantile',
    threshold_quantile=0.95,
    thresholds_path=str(THRESHOLDS_PATH) if THRESHOLDS_PATH else None,
    max_samples_per_category=max_samples,
    resume=True,
    attack=attack,
    target_kwargs={'clip_download_root': '/kaggle/working/clip_cache'},
)

print('===== STEP 4: RUN EXPERIMENT =====')
print('Mode:', 'FULL' if FULL_RUN else 'SMOKE')
print('Checkpoint:', CHECKPOINT_PATH)
print('Output:', OUTPUT_ROOT)
summary_path = run_experiment(config)
print('Finished. Summary:', summary_path)

In [ ]:
import shutil
from pathlib import Path

print('===== STEP 5: PACKAGE OUTPUTS =====')
output_root = OUTPUT_ROOT
if not output_root.is_dir():
    raise FileNotFoundError(f'Output directory was not created: {output_root}')
archive = shutil.make_archive(
    str(output_root),
    'zip',
    root_dir=output_root.parent,
    base_dir=output_root.name,
)
print('Packaged results:', archive)